# 🧠 NLA Hallucination Detection Pipeline (v2 — Raw Vector Protocol)
**Senior Project — SIIT Thammasat University**  
**Model:** Qwen2.5-7B-Instruct | **Dataset:** HaluEval QA + In-dist validation | **NLA Layer:** 20

| Phase | Description | VRAM |
|-------|-------------|------|
| 0 — Setup | Install deps + download models | Low |
| Session 1A | Extract Layer 20 activations — HaluEval (Qwen only) | ~14 GB |
| Session 1B | Extract Layer 20 activations — In-dist (WildChat + Ultra-FineWeb) | ~14 GB |
| Session 2 | NLA Verbalization (AV) + AR Scoring — both datasets | ~40 GB |

> ⚠️ **Session 1 and Session 2 must run in separate kernels** (Qwen + SGLang > 48 GB combined)

### 🔄 Changes from v1 (per Qwen NLA validation protocol)
1. **No L2-normalization at rest** — vectors saved as raw float32; scaling happens at inference via `injection_scale`
2. **MIN_POSITION = 50** — only sample tokens at index ≥ 50 (early tokens = attention sinks / immature features)
3. **Greedy decoding** at AV (deterministic, temperature = 0)
4. **New Session 1B** — in-distribution validation set (50% WildChat-1M + 50% Ultra-FineWeb) to reproduce the 0.752 fve_nrm baseline and prove the pipeline is correct

## 🔧 0. Environment Check

In [ ]:
import torch

print(f'PyTorch:    {torch.__version__}')
print(f'CUDA:       {torch.cuda.is_available()}')
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print(f'GPU:        {props.name}')
    print(f'Total VRAM: {props.total_memory / 1e9:.1f} GB')
    print(f'Used  VRAM: {torch.cuda.memory_allocated() / 1e9:.1f} GB')
!nvidia-smi | grep MiB

## 📦 1. Install Dependencies

In [ ]:
!pip install huggingface_hub accelerate datasets scikit-learn tqdm -q
!pip install "sglang[all]==0.5.6" -q
!cd /workspace/natural_language_autoencoders && pip install -e . -q
print('✅ All dependencies installed!')

## 📥 2. Download Models
> ⚠️ `allenai/WildChat-1M` is a **gated dataset** — accept the license on HuggingFace and login with `huggingface-cli login` (or set `HF_TOKEN`) before Session 1B.

In [ ]:
import os
os.environ.update({
    'HF_HOME': '/workspace/hf_cache',
    'TRANSFORMERS_CACHE': '/workspace/hf_cache',
})

from huggingface_hub import snapshot_download

MODELS = {
    'Qwen/Qwen2.5-7B-Instruct':    '/workspace/models/qwen2.5-7b-instruct',
    'kitft/nla-qwen2.5-7b-L20-av': '/workspace/models/nla-av',
    'kitft/nla-qwen2.5-7b-L20-ar': '/workspace/models/nla-ar',
}

for repo_id, local_dir in MODELS.items():
    if os.path.exists(local_dir) and os.listdir(local_dir):
        print(f'⏩ Skip (exists): {local_dir}')
        continue
    print(f'📥 Downloading {repo_id}...')
    snapshot_download(repo_id=repo_id, local_dir=local_dir)
    print(f'✅ Done: {local_dir}')

# Verify
print()
for path in MODELS.values():
    n = len(os.listdir(path)) if os.path.exists(path) else 0
    print(f'  {"✅" if n > 0 else "❌"} {path.split("/")[-1]:35s} ({n} files)')
print('\n✅ All models ready!')

---
## 🧠 SESSION 1A — Extract Layer 20 Activations (HaluEval, RAW)

**Protocol changes vs v1:**
- ❌ No L2-normalization — save **raw residual stream vectors** (float32)
- ✅ Record last-token position; flag samples where position < `MIN_POSITION` (50)
- Hook on `model.model.layers[20]` output = residual stream **after** block 20, **before** final norm ✅

> **After running all Session 1 cells: restart the kernel before Session 2!**

In [ ]:
import torch
import numpy as np
from transformers import AutoTokenizer, AutoModelForCausalLM
from datasets import load_dataset
from tqdm import tqdm

# ─── Config ─────────────────────────────────────
MODEL_PATH   = '/workspace/models/qwen2.5-7b-instruct'
LAYER_IDX    = 20
N_SAMPLES    = 200
MIN_POSITION = 50          # protocol: tokens at index >= 50 only
SAVE_DIR     = '/workspace'

# ─── Load Qwen ──────────────────────────────────
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH,
    torch_dtype=torch.bfloat16,
    device_map='auto',
    low_cpu_mem_usage=True,
)
model.eval()
print(f'✅ Qwen loaded | VRAM: {torch.cuda.memory_allocated()/1e9:.1f} GB')

# ─── Load HaluEval ──────────────────────────────
dataset = load_dataset('pminervini/HaluEval', 'qa_samples', split='data')
samples = dataset.select(range(N_SAMPLES))
print(f'✅ HaluEval: {len(samples)} samples')

In [ ]:
# ─── Shared extraction utilities (used by 1A and 1B) ──
_captured = {}

def hook_fn(module, input, output):
    hidden = output[0] if isinstance(output, tuple) else output
    # Keep FULL sequence hidden states; raw, no normalization
    _captured['h'] = hidden.detach().float().cpu()   # [1, seq, hidden]

hook = model.model.layers[LAYER_IDX].register_forward_hook(hook_fn)

@torch.no_grad()
def extract_at_position(text: str, position: int = -1, max_length: int = 1024):
    """Run forward pass, return (raw_vector[hidden], token_position, seq_len)."""
    inputs = tokenizer(text, return_tensors='pt', truncation=True,
                       max_length=max_length).to('cuda')
    model(**inputs)
    seq_len = _captured['h'].shape[1]
    pos = seq_len - 1 if position == -1 else min(position, seq_len - 1)
    return _captured['h'][0, pos, :].clone(), pos, seq_len

In [ ]:
activations_list, labels_list, positions_list = [], [], []

for sample in tqdm(samples, desc=f'HaluEval L{LAYER_IDX} extraction (raw)'):
    prompt = f"Question: {sample['question']}\nAnswer: {sample['answer']}"
    vec, pos, seq_len = extract_at_position(prompt, position=-1, max_length=512)
    activations_list.append(vec)
    positions_list.append(pos)
    labels_list.append(1 if sample['hallucination'] == 'yes' else 0)

activations_np = torch.stack(activations_list).numpy().astype(np.float32)  # RAW
labels_np      = np.array(labels_list)
positions_np   = np.array(positions_list)

np.save(f'{SAVE_DIR}/activations_L20_raw.npy', activations_np)
np.save(f'{SAVE_DIR}/labels_L20.npy',          labels_np)
np.save(f'{SAVE_DIR}/positions_L20.npy',       positions_np)

n_short = (positions_np < MIN_POSITION).sum()
print(f'\n✅ Saved RAW activations: {activations_np.shape} | dtype: {activations_np.dtype}')
print(f'   Mean vector norm: {np.linalg.norm(activations_np, axis=-1).mean():.2f} (should be >> 1 if raw)')
print(f'   Hallucinated: {labels_np.sum()} | Truthful: {(labels_np==0).sum()}')
print(f'   ⚠️  Samples with last-token position < {MIN_POSITION}: {n_short}/{len(positions_np)}')
print(f'      (short prompts = immature features; report/filter these in analysis)')

---
## 🌐 SESSION 1B — In-Distribution Validation Set (WildChat + Ultra-FineWeb)

**Goal:** reproduce the in-dist baseline (`fve_nrm ≈ 0.752`, cosine ≈ 0.890) to prove the pipeline is correct.

**Protocol:**
- 50% `allenai/WildChat-1M` (conversational) + 50% `openbmb/Ultra-FineWeb` (web text)
- Sample ONE token per document at random position `>= MIN_POSITION` (50)
- Documents must be long enough: `seq_len > MIN_POSITION`
- Raw vectors, no normalization

> Uses `streaming=True` so we don't download the full corpora.
> Note: we can't guarantee zero doc-overlap with Anthropic's training split, but random samples from 1M+ docs are overwhelmingly likely unseen — document this caveat in the thesis.

In [ ]:
import random
random.seed(42)

N_INDIST_PER_SOURCE = 100   # 100 WildChat + 100 Ultra-FineWeb = 200 total
MAX_LEN_INDIST      = 1024

def wildchat_texts(n):
    """Yield flattened conversation texts from WildChat-1M (gated — needs HF login)."""
    ds = load_dataset('allenai/WildChat-1M', split='train', streaming=True)
    count = 0
    for row in ds:
        try:
            conv = row['conversation']
            text = '\n'.join(f"{t['role']}: {t['content']}" for t in conv)
        except Exception:
            continue
        if len(text) < 400:      # cheap pre-filter before tokenizing
            continue
        yield text
        count += 1
        if count >= n * 3:       # over-yield; we filter by token length later
            break

def ultrafineweb_texts(n):
    """Yield web documents from Ultra-FineWeb (English config)."""
    ds = load_dataset('openbmb/Ultra-FineWeb', 'en', split='train', streaming=True)
    count = 0
    for row in ds:
        text = row.get('content') or row.get('text') or ''
        if len(text) < 400:
            continue
        yield text
        count += 1
        if count >= n * 3:
            break

def extract_indist(text_iter, n_target, desc):
    vecs, poss = [], []
    pbar = tqdm(total=n_target, desc=desc)
    for text in text_iter:
        if len(vecs) >= n_target:
            break
        # Tokenize first to check length without running the model
        ids = tokenizer(text, truncation=True, max_length=MAX_LEN_INDIST)['input_ids']
        if len(ids) <= MIN_POSITION + 1:
            continue                              # too short — skip
        pos = random.randint(MIN_POSITION, len(ids) - 1)
        vec, actual_pos, _ = extract_at_position(text, position=pos,
                                                 max_length=MAX_LEN_INDIST)
        vecs.append(vec)
        poss.append(actual_pos)
        pbar.update(1)
    pbar.close()
    return vecs, poss

wc_vecs, wc_pos = extract_indist(wildchat_texts(N_INDIST_PER_SOURCE),
                                 N_INDIST_PER_SOURCE, 'WildChat')
uf_vecs, uf_pos = extract_indist(ultrafineweb_texts(N_INDIST_PER_SOURCE),
                                 N_INDIST_PER_SOURCE, 'Ultra-FineWeb')

indist_np = torch.stack(wc_vecs + uf_vecs).numpy().astype(np.float32)
source_np = np.array([0]*len(wc_vecs) + [1]*len(uf_vecs))  # 0=WildChat, 1=UFW

np.save(f'{SAVE_DIR}/activations_indist_raw.npy', indist_np)
np.save(f'{SAVE_DIR}/sources_indist.npy',         source_np)

print(f'\n✅ In-dist set: {indist_np.shape} | positions all >= {MIN_POSITION} ✅')
print(f'   WildChat: {len(wc_vecs)} | Ultra-FineWeb: {len(uf_vecs)}')
print(f'   Mean vector norm: {np.linalg.norm(indist_np, axis=-1).mean():.2f}')

In [ ]:
import gc

hook.remove()
del model
torch.cuda.empty_cache()
gc.collect()

print(f'✅ Qwen freed | VRAM: {torch.cuda.memory_allocated()/1e9:.1f} GB')
print('\n⚠️  Restart the kernel now before running Session 2!')

---
## 💬 SESSION 2 — NLA Verbalization + AR Scoring

### ⚠️ Prerequisites
1. Kernel must be **fresh** (Qwen not loaded)
2. SGLang AV server must be running:
```bash
python -m sglang.launch_server \
    --model /workspace/models/nla-av \
    --port 30000 \
    --mem-fraction-static 0.85 \
    --disable-radix-cache \
    --trust-remote-code \
    --dtype bfloat16
```
Wait for server ready message before continuing.

**Protocol:** verbalization must use **deterministic greedy decoding** (temperature = 0). Check `NLAClient.generate()` signature in `nla_inference.py` — if it accepts sampling params, pass `temperature=0.0` explicitly; the cell below tries this and falls back if unsupported.

In [ ]:
import sys, inspect, torch, numpy as np
sys.path.insert(0, '/workspace/natural_language_autoencoders')
from nla_inference import NLAClient, NLACritic

# ─── Load saved RAW activations ─────────────────────
halu_acts = np.load('/workspace/activations_L20_raw.npy')
labels    = np.load('/workspace/labels_L20.npy')
ind_acts  = np.load('/workspace/activations_indist_raw.npy')
print(f'✅ HaluEval: {halu_acts.shape} | In-dist: {ind_acts.shape}')

# ─── Init NLA clients ─────────────────────────────
client = NLAClient(
    checkpoint_dir='/workspace/models/nla-av',
    sglang_url='http://127.0.0.1:30000'
)
critic = NLACritic(
    checkpoint_dir='/workspace/models/nla-ar',
    device='cuda',
    dtype=torch.bfloat16
)

# ─── Greedy decoding wrapper ────────────────────────
_gen_params = inspect.signature(client.generate).parameters
GREEDY_KW = {}
for k, v in [('temperature', 0.0), ('top_p', 1.0)]:
    if k in _gen_params:
        GREEDY_KW[k] = v
print(f'Greedy kwargs supported by client.generate: {GREEDY_KW or "NONE — check nla_inference.py / server defaults!"}')

def verbalize(vec):
    return client.generate(vec, **GREEDY_KW)

# ─── Sanity check: single sample ────────────────────
v = halu_acts[0]
explanation = verbalize(v)
mse, cos    = critic.score(explanation, v)
print(f'\n--- Single-sample test (HaluEval idx=0) ---')
print(f'Verbalization:\n{explanation}')
print(f'MSE: {mse:.4f} | Cosine: {cos:.4f}')
# Determinism check (greedy => identical output)
assert verbalize(v) == explanation, '⚠️ Non-deterministic output — greedy decoding NOT active!'
print('✅ Deterministic (greedy) decoding confirmed')

In [ ]:
from tqdm import tqdm

# ─── Full verbalization (AV) — both datasets ──────────
def run_av(acts, name):
    exps = [verbalize(v) for v in tqdm(acts, desc=f'AV: {name}')]
    np.save(f'/workspace/explanations_{name}.npy', np.array(exps, dtype=object))
    print(f'✅ Saved {len(exps)} → /workspace/explanations_{name}.npy')
    return exps

exps_indist = run_av(ind_acts,  'indist')    # ⭐ run in-dist FIRST (validates pipeline)
exps_halu   = run_av(halu_acts, 'halueval')

In [ ]:
from tqdm import tqdm

# ─── AR Reconstruction + metrics ─────────────────────
def evaluate(exps, acts, name, fve_target, cos_target):
    preds = [critic.reconstruct(e) for e in tqdm(exps, desc=f'AR: {name}')]
    preds = torch.stack(preds)

    mse_scale = critic.mse_scale
    gold   = torch.tensor(acts, dtype=torch.float32)
    gold_n = gold  / gold.norm(dim=-1, keepdim=True)  * mse_scale
    pred_n = preds / preds.norm(dim=-1, keepdim=True) * mse_scale

    mu  = gold_n.mean(dim=0)
    fve = 1 - ((pred_n - gold_n)**2).mean() / ((gold_n - mu)**2).mean()
    cos = torch.nn.functional.cosine_similarity(pred_n, gold_n, dim=-1)

    print(f'\n{"="*52}')
    print(f'  [{name}]')
    print(f'  fve_nrm  : {fve.item():.4f}   (target: ~{fve_target})')
    print(f'  Mean cos : {cos.mean().item():.4f}   (target: ~{cos_target})')
    print(f'  Mean MSE : {(2*(1-cos)).mean().item():.4f}')
    print(f'{"="*52}')
    return fve.item(), cos

# ⭐ In-dist first: if this hits ~0.75 / ~0.89, the pipeline is validated
fve_ind,  cos_ind  = evaluate(exps_indist, ind_acts,  'IN-DIST (WildChat+UFW)', 0.752, 0.890)
fve_halu, cos_halu = evaluate(exps_halu,   halu_acts, 'HaluEval QA (OOD)',      '—',   '—')

print('\n[Interpretation]')
print('- In-dist ≈ targets  → pipeline correct; HaluEval gap = genuine distribution shift')
print('- In-dist << targets → check: raw vectors? greedy decoding? paraphrase-ceiling')
print('  normalization in repo eval scripts? injection_scale=150?')

results = {
    'fve_indist': fve_ind, 'cos_indist': cos_ind.mean().item(),
    'fve_halueval': fve_halu, 'cos_halueval': cos_halu.mean().item(),
}
import json
with open('/workspace/results_v2.json', 'w') as f:
    json.dump(results, f, indent=2)
print('\n✅ Saved → /workspace/results_v2.json')

---
## 📋 Next Steps
1. **Check paraphrase ceiling** — raw FVE must be divided by the ceiling from the repo's eval scripts to get true `fve_nrm`; search `natural_language_autoencoders/` for it before comparing to 0.752
2. **Semantic theme analysis** — compare verbalization themes: hallucinated vs. truthful (use `explanations_halueval.npy` + `labels_L20.npy`)
3. **Re-run linear probe** on raw (un-normalized) L20 vectors as a bonus comparison
4. **Download results** from `/workspace/` → terminate pod

Key saved files:
- `/workspace/activations_L20_raw.npy` — HaluEval raw L20 activations [200, 3584]
- `/workspace/positions_L20.npy` — token positions (flag < 50)
- `/workspace/activations_indist_raw.npy` — in-dist raw vectors [200, 3584]
- `/workspace/sources_indist.npy` — 0 = WildChat, 1 = Ultra-FineWeb
- `/workspace/explanations_indist.npy`, `/workspace/explanations_halueval.npy`
- `/workspace/results_v2.json`